In [10]:
# 필수 라이브러리 설치 (최초 1회 실행)
!pip install pandas scikit-learn boto3 transformers torch python-dotenv --quiet

## 1. 데이터 로딩 및 전처리
- 로컬 data/ 폴더에서 원본 데이터 불러오기
- 결측치/이상치 처리, 토큰화, 정제
- 훈련/검증/테스트 분할
- (Best Practice)
  - 원본 데이터는 그대로 두고, 전처리/분할된 데이터만 S3에 저장 (불필요한 중복 방지)

In [11]:
# 환경 변수에서 AWS 자격증명 로드 (프로젝트 루트 .env.mlu 사용)
from dotenv import load_dotenv
load_dotenv('../.env.mlu')

True

In [ ]:
import os
from dotenv import load_dotenv

# Try loading the .env.mlu file and print result
env_loaded = load_dotenv('.env.mlu')
print("dotenv loaded:", env_loaded)
print("AWS_ACCESS_KEY_ID:", os.getenv("AWS_ACCESS_KEY_ID"))
print("AWS_SECRET_ACCESS_KEY:", os.getenv("AWS_SECRET_ACCESS_KEY"))

# Advanced diagnostic: search for .env.mlu in all likely locations
import os
from glob import glob

print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir())

# Search for .env.mlu in common locations
search_paths = [
    ".env.mlu",
    "./.env.mlu",
    "../.env.mlu",
    "/home/jovyan/work/.env.mlu",
    "/workspace/mlu/.env.mlu",
    "/workspace/.env.mlu",
    "/.env.mlu"
]
found = False
for path in search_paths:
    if os.path.exists(path):
        print(f"Found: {path}")
        with open(path) as f:
            print(f"Contents of {path}:\n", f.read())
        found = True
if not found:
    print(".env.mlu not found in any common location.")

# List all .env.mlu files in the entire filesystem (may be slow)
for file in glob('/**/.env.mlu', recursive=True):
    print(f"Found by glob: {file}")

In [15]:
import os
import pandas as pd
import boto3
# 데이터 로딩 (이미 전처리된 데이터 사용)
data_path = './data/preprocessed.csv'
df = pd.read_csv(data_path)
# 데이터 분할 (예시)
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2, random_state=42)
# S3에 저장 (Best Practice: 처리된 데이터만)
s3 = boto3.client('s3')
bucket_name = 'elbee-oreumi'  # 실제 S3 버킷명으로 변경 필요
try:
    train.to_csv('train_processed.csv', index=False)
    test.to_csv('test_processed.csv', index=False)
    s3.upload_file('train_processed.csv', bucket_name, 'processed/train_processed.csv')
    s3.upload_file('test_processed.csv', bucket_name, 'processed/test_processed.csv')
except Exception as e:
    print(f'⚠️ S3 업로드 오류: {e}\n버킷 이름, 권한, 네트워크를 확인하세요.')

c:\Users\hsyyu\anaconda3\envs\mlu\lib\site-packages\boto3\compat.py:84: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [17]:
# 원본 데이터와 분할된 데이터 비교 (샘플)
import pandas as pd

# 원본 데이터 로드
df_source = pd.read_csv('./data/preprocessed.csv')
print('원본 데이터 shape:', df_source.shape)
print('원본 데이터 샘플:')
display(df_source.head())

# train/test 데이터 로드
train_df = pd.read_csv('train_processed.csv')
test_df = pd.read_csv('test_processed.csv')
print('\n[Train] shape:', train_df.shape)
print('[Train] 샘플:')
display(train_df.head())
print('\n[Test] shape:', test_df.shape)
print('[Test] 샘플:')
display(test_df.head())

# 컬럼별 차이 확인
print('\n컬럼 차이:', set(df_source.columns) ^ set(train_df.columns))

# 중복/누락 샘플 체크
print('\nTrain/Test 중복 샘플 수:', len(set(train_df.index) & set(test_df.index)))

원본 데이터 shape: (26871, 2)
원본 데이터 샘플:


,리뷰,평점
0,선물용으로 빨리 받아서 전달했어야하는 상품이었는데 머그 컵만 와서 당황했습니다. 전...,0
1,주문을 11월 6에 시켰는데 11월 16일에 배송이 왔네요 ㅎㅎㅎ 여기 회사 측과는...,0
2,재구매 늘 사는 흙이에요 팽이들이 젤 좋아해요 빠른 배송 감사합니다,1
3,"배송 기사나, 판매하는 회사나 불친절하고 불쾌합니다. 서비스 면에선 최악이네요 제품...",0
4,진짜 너무 하시네요 배송이 늦는 건 엄 절 수 없다 하고 참 앗지만 밑에 서랍이 금...,0



[Train] shape: (21496, 2)
[Train] 샘플:


,리뷰,평점
0,배송이 너무 느려요 한 달 정도 걸린 듯요..,0
1,배송이 빨라서 좋았어요 급한 물건이었는데 감사합니다.,1
2,좋습니다 배송은 소 소 합 니자,1
3,배송 포장에 신경 좀 써 주셨으면하네요.,0
4,나이쑤 포장 튼튼히 잘 해 주셨고 잘 되네요~~ 다만 공인 인증서 때문에 보안 프로...,1



[Test] shape: (5375, 2)
[Test] 샘플:


,리뷰,평점
0,빠른 배송 너무 젛 아요!!!,1
1,배송 안전하게 잘 도착했습니다,0
2,아버지께서 아보카도 첫 맛을 보고 너무 좋아하십니다. 싱싱하고 빠르게 배송되어 잘 ...,1
3,배송 착오가 있어 오지 않았는데 다시 보내 주셔서 감사 드립니다. 닭 목살 소금 구...,1
4,배송 빨랐고 디자인 정말 귀엽고 예쁘요 조용해요 다만 성능은 다소 떨어집니다 작은 ...,1



컬럼 차이: set()

Train/Test 중복 샘플 수: 5375


### 데이터 파이프라인 설명 및 AWS 활용 방식

#### 용어 및 메서드 정의

- **DataFrame**: pandas의 2차원 데이터 구조로, 표 형태의 데이터를 다룸.
- **train_test_split**: scikit-learn의 함수로, 데이터를 훈련/테스트 세트로 분할.
- **to_csv**: pandas DataFrame을 CSV 파일로 저장하는 메서드.
- **boto3.client('s3')**: AWS S3와 상호작용하는 파이썬 라이브러리의 클라이언트 객체 생성.
- **upload_file**: boto3의 S3 클라이언트 메서드로, 로컬 파일을 S3 버킷에 업로드.

#### 파이프라인 동작 요약

1. 전처리된 데이터를 불러와 DataFrame으로 로드합니다.
2. 데이터를 훈련/테스트 세트로 분할합니다.
3. 분할된 데이터를 각각 CSV 파일로 저장합니다.
4. 이 파일들을 AWS S3 버킷의 `processed/` 폴더에 업로드합니다.

#### AWS 파이프라인에서의 활용
- 이 방식은 **데이터 준비 및 업로드 자동화**의 기초 단계입니다.
- S3에 업로드된 데이터는 이후 AWS의 다양한 서비스(예: SageMaker, Lambda, Glue, EMR 등)에서 바로 접근하여 사용할 수 있습니다.
- 예시: SageMaker에서 학습 작업을 생성할 때, S3의 `processed/train_processed.csv`와 `processed/test_processed.csv` 경로를 입력하면 별도의 데이터 이동 없이 바로 학습에 활용할 수 있습니다.
- 이 구조는 **데이터 파이프라인의 표준화**와 **재현성**을 높여주며, 협업 및 자동화된 MLOps 환경 구축의 기반이 됩니다.

In [16]:
# S3 버킷에 저장된 파일 목록 확인
try:
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix='processed/')
    if 'Contents' in response:
        print('S3 버킷 내 processed/ 폴더 파일 목록:')
        for obj in response['Contents']:
            print(obj['Key'], f"({obj['Size']} bytes)")
    else:
        print('S3 버킷에 processed/ 폴더가 비어있거나 파일이 없습니다.')
except Exception as e:
    print(f'⚠️ S3 목록 조회 오류: {e}')

S3 버킷 내 processed/ 폴더 파일 목록:
processed/test_processed.csv (663384 bytes)
processed/train_processed.csv (2647089 bytes)
